# IM-Benchmark-Lab — Google Colab Setup

Notebook này cài đặt và chạy dự án **IM-Benchmark-Lab** trên Google Colab.

## Nội dung

1. Kiểm tra môi trường
2. Cài system dependencies
3. Lấy source code
4. Tạo venv + cài thư viện Python
5. Build C++ core (pybind11)
6. Smoke test Python ↔ C++
7. Chạy unit tests
8. Chạy thí nghiệm mẫu
9. Tùy chọn: tải trước dữ liệu
10. Tùy chọn: GUI smoke test
11. Tùy chọn: lưu kết quả vào Google Drive
12. Chạy lại thí nghiệm với tham số tùy chỉnh

## Cách dùng

- Chọn **Runtime → Change runtime type → CPU** (dự án không cần GPU).
- Chạy lần lượt các cell từ trên xuống.
- Lần chạy đầu có thể mất 3–10 phút để cài thư viện và build C++.

> **Lưu ý:** Colab không có desktop thật, nên GUI CustomTkinter không mở được trực tiếp. Notebook chỉ chạy GUI smoke test bằng Xvfb. Muốn dùng GUI, chạy trên WSL2/Windows.


## 0. Kiểm tra môi trường


In [ ]:
!python3 --version
!cmake --version | head -n 1
!g++ --version | head -n 1
!nproc
!df -h /content | tail -n 1


## 1. Cài system dependencies

Cài `git`, `cmake`, `g++`, Python headers, Tk và OpenMP runtime.


In [ ]:
%%bash
set -Eeuo pipefail

export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq \
    git \
    cmake \
    build-essential \
    g++ \
    python3-dev \
    python3-venv \
    python3-tk \
    libgomp1

echo "System dependencies OK"
cmake --version | head -n 1
g++ --version | head -n 1
python3 --version


## 2. Lấy source code

Clone repo (hoặc dùng lại nếu đã có).


In [ ]:
%%bash
set -Eeuo pipefail

REPO_URL="https://github.com/SLMR05-05/influence-maximization-benchmark.git"
BRANCH="main"
PROJECT_DIR="/content/influence-maximization-benchmark"

if [ -f "$PROJECT_DIR/pyproject.toml" ] && [ -d "$PROJECT_DIR/src/im_research" ]; then
    echo "Đã có source: $PROJECT_DIR"
else
    git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
fi

git config --global --add safe.directory "$PROJECT_DIR" || true
mkdir -p "$PROJECT_DIR/data/raw" "$PROJECT_DIR/data/processed" "$PROJECT_DIR/runs"
cd "$PROJECT_DIR"
echo "Commit: $(git rev-parse --short HEAD 2>/dev/null || echo 'n/a')"


## 3. Tạo môi trường ảo + cài thư viện

Venv đặt tại `/content/im_env` để không lẫn với source.


In [ ]:
%%bash
set -Eeuo pipefail

PROJECT_DIR="/content/influence-maximization-benchmark"
VENV_DIR="/content/im_env"

if [ ! -x "$VENV_DIR/bin/python" ]; then
    python3 -m venv "$VENV_DIR"
fi

source "$VENV_DIR/bin/activate"
python -m pip install --upgrade pip setuptools wheel
python -m pip install --no-cache-dir -r "$PROJECT_DIR/requirements.txt"
python -m pip install --no-cache-dir --no-build-isolation -e "$PROJECT_DIR"

python --version
echo "Python dependencies OK"


## 4. Build C++ core

Build `_im_core*.so` và đặt trực tiếp vào `src/im_research/`.


In [ ]:
%%bash
set -Eeuo pipefail

PROJECT_DIR="/content/influence-maximization-benchmark"
cd "$PROJECT_DIR"

rm -rf build
mkdir -p build
cd build

cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CXX_FLAGS="-O3 -march=native -fopenmp"

make -j"$(nproc)"

cd "$PROJECT_DIR"
find src/im_research -maxdepth 1 -name '_im_core*.so' -ls


## 5. Smoke test Python ↔ C++


In [ ]:
%%bash
set -Eeuo pipefail
source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python - <<'PY'
import im_research
from im_research import _im_core

print(f"im_research : {im_research.__file__}")
print(f"C++ core    : {_im_core.__file__}")

g = _im_core.Graph.from_edge_list(
    3, [(0, 1), (1, 2)], [0.5, 0.5], False
)
print(f"Graph test  : {g.num_nodes()} đỉnh, {g.num_edges()} cạnh")

ev = _im_core.Evaluator(
    g, _im_core.DiffusionModel.IndependentCascade,
    num_mc_simulations=50, seed=42,
)
print(f"IC spread   : {ev.evaluate([0])}")
PY


## 6. Chạy unit tests


In [ ]:
%%bash
set -Eeuo pipefail
source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python -m pytest tests/unit -v


## 7. Chạy thí nghiệm mẫu

Chạy 6 thuật toán trên Karate Club với `k=10`, `R_search=200`, `R_val=1000`.


In [ ]:
%%bash
set -Eeuo pipefail
source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python -m im_research.experiments.runner \
    --dataset "Karate Club" \
    --model IC \
    --p 0.1 \
    --k 10 \
    --mc 200 \
    --mc_val 1000 \
    --runs 1 \
    --algo iga \
    --algo ga \
    --algo celf \
    --algo degree_discount \
    --algo pagerank \
    --algo degree


## 8. Tùy chọn — tải trước 5 bộ dữ liệu

`data/raw/` không được commit lên Git. Cell này tải trước để lần chạy sau nhanh hơn.


In [ ]:
%%bash
set -Eeuo pipefail
source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python - <<'PY'
from im_research.data.loader import load_networkx_graph

datasets = [
    "Karate Club",
    "Dolphins",
    "ego-Facebook",
    "NetHEPT",
    "Email-Enron",
]
for name in datasets:
    try:
        g, meta = load_networkx_graph(name)
        print(f"{name:15s} {g.number_of_nodes():>6} đỉnh / {g.number_of_edges():>7} cạnh")
    except Exception as exc:
        print(f"{name:15s} LỖI: {exc}")
PY


## 9. Tùy chọn — GUI smoke test (Xvfb)

Colab không có desktop; cell này chỉ kiểm tra GUI khởi động được, không hiển thị cửa sổ.


In [ ]:
%%bash
set -Eeuo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get install -y -qq xvfb xauth

source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

xvfb-run -a -s "-screen 0 1400x900x24" python - <<'PY'
from im_research.gui.app import IMApp

app = IMApp()
app.after(3000, app.destroy)
app.mainloop()
print("GUI smoke test OK")
PY


## 10. Tùy chọn — lưu kết quả vào Google Drive

Cell này dùng `google.colab`, chỉ chạy được trên Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

src = Path('/content/influence-maximization-benchmark/runs')
dst = Path('/content/drive/MyDrive/IM-Benchmark-Lab/runs')
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Đã lưu kết quả vào', dst)
else:
    print('Chưa có thư mục runs/. Hãy chạy thí nghiệm trước.')


## 11. Chạy lại thí nghiệm với tham số tùy chỉnh

Sửa các biến ở đầu cell rồi chạy.


In [ ]:
%%bash
set -Eeuo pipefail
source /content/im_env/bin/activate
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

# ── Tham số có thể sửa ──────────────────────────────────────
DATASET="Karate Club"
MODEL="IC"
P="0.1"
K="10"
MC="200"
MC_VAL="1000"
RUNS="3"
ALGOS="iga ga celf degree_discount pagerank degree"
# ────────────────────────────────────────────────────────────

ALGO_ARGS=""
for a in $ALGOS; do
    ALGO_ARGS="$ALGO_ARGS --algo $a"
done

python -m im_research.experiments.runner \
    --dataset "$DATASET" \
    --model "$MODEL" \
    --p "$P" \
    --k "$K" \
    --mc "$MC" \
    --mc_val "$MC_VAL" \
    --runs "$RUNS" \
    $ALGO_ARGS


## Ghi chú

- **GUI:** Colab không có desktop. Muốn dùng GUI thật, chạy `python src/im_research/gui/main_app.py` trên WSL2/Windows.
- **CLI:** dùng `python -m im_research.experiments.runner`; entry point `im-bench` trong `pyproject.toml` hiện trỏ tới module chưa tồn tại.
- **Dữ liệu:** `data/raw/` không nằm trong Git; loader tự tải Facebook, Enron, NetHEPT từ Internet. Karate lấy từ NetworkX; Dolphins tải từ UCI.
- **Kết quả:** `runs/EXP_<timestamp>/summary.json` và `runs/latest_results.json`.
- **Cập nhật code:** `!git -C /content/influence-maximization-benchmark pull --ff-only origin main`
- **Cấu hình IC:** `p=0.1`, `R_search=200`, `R_val=1000`; đổi bằng `--p`, `--mc`, `--mc_val`.
